In [ ]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
import random
import sys
from pathlib import Path
print(str(Path().resolve().parents[1]))
from paths import *


In [ ]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [ ]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    Create, configure, train, and export an instance segmentation model using a configurable setup.

    This function prepares the dataset, configures the training pipeline, trains the model for the
    specified number of epochs, and exports the resulting model to both PyTorch and ONNX formats.

    Parameters:
    -----------
    trainDirectory : str
        Path to the training dataset folder.
    testDirectory : str
        Path to the testing dataset folder.
    modelName : str
        Name assigned to the trained model and saved output files.
    epochs : int
        Number of training epochs.
    labels : list[str]
        List of label names used for segmentation. For example, ['parking_spaces'].
    augment_data : bool
        Whether to apply data augmentation during training.
    save_path : str
        Directory path where the model will be saved.
    save_interval : int, optional
        Interval (in epochs) at which to save model checkpoints. Default is 0 (disabled).
    maskdata : list[float], optional
        List containing three float values:
        [scoreThreshold, maskThreshold, strideFraction] used for ONNX metadata.
        Defaults to [0.2, 0.3, 0.5] if not provided.
    model_description : str, optional
        Description to embed into the ONNX metadata. Default is "default".

    Returns:
    --------
    model : torch.nn.Module
        The trained model set to evaluation mode.
    config : Configuration
        The configuration object used for training and exporting the model.

    Notes:
    ------
    - The function prints dataset statistics and model file names for verification.
    - Legend entries are dynamically created based on the provided `labels` list.
    - If `augment_data` is True, data augmentation is applied during training using `createTransforms(True)`.
    - The function validates dataset consistency before training.
    - After training, the model is saved and exported to ONNX, and metadata is written.
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [ ]:
train_directory = TRAIN
test_directory = TEST
modelName = "<insert name here>"
epochs = 1
labels = ["parking_space"]
augment = False
save_path = MODELS_DIR
save_Interval = 0 # model is saved in between these amount of epochs
model, config = createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

^ AP and AR converge to the results at the last epoch and don't change after with the current dataset.